# ANN API Flow Example

This example demonstrates how ANN (Approximate Nearest Neighbor) can be performed with enVector,
by running APIs explicitly as in the API flow example.

IVF (Inverted File) is one of the most popular indexing algorithms to enhance search efficiency.
The enVector supports IVF with **enVector-customized indexing algorithm, called `IVF_VCT`** for encrypted similarity search.
In this notebook, we combine ANN index preparation with the explicit encrypted search flow.


## Import SDK

Before we start, we should install and import the `pyenvector` package to use enVector Python APIs.


In [ ]:
# !pip install scikit-learn

import pyenvector as ev
import numpy as np


## Prepare Dataset

First, we generate a large sample dataset for test purposes.


In [ ]:
num_vectors = 10_000
dim = 1536

vectors = np.random.rand(num_vectors, dim).astype(np.float32)
vectors = vectors / np.linalg.norm(vectors, axis=1, keepdims=True)
metadata = [f"Metadata-{i+1}" for i in range(num_vectors)]


### Create Centroids

To perform IVF-VCT, we train the centroids with the given dataset by running k-means clustering.
The `nlist` parameter indicates the number of clusters in an IVF index.


In [ ]:
from sklearn.cluster import KMeans
# for GPU acceleration, we can use cuml.cluster.KMeans

nlist = 50

kmeans = KMeans(n_clusters=nlist, n_init=1)
kmeans.fit(vectors)
centroids = kmeans.cluster_centers_.copy()
centroids_idx = kmeans.predict(vectors).tolist()


## Vector Search

### 1. Initialize

To use the enVector service, initialization is required to connect to the server and register keys.
Here, we keep each step explicit (`init_connect`, key generation, key registration) as in API flow.


In [ ]:
import os
address = os.getenv("ENVECTOR_ADDRESS", "localhost:50050")
key_path = "./keys"
key_id = "ann_api_flow_key"
index_name = "ann_api_flow_index"

# Connect to the enVector end-point server with address and access token (if required)
ev.init_connect(
    address=address,
    # access_token="...",  # if needed
)

### 2. Generate and Register Key

Generate FHE key files on the client, configure index key settings, then register/load the key on server.


In [ ]:
from pyenvector.crypto import KeyGenerator

# Generate keys locally
keygen = KeyGenerator(
    key_path=f"{key_path}/{key_id}",
    eval_mode="mms32",
    preset="ip2",
    metadata_encryption=True,
)
keygen.generate_keys()

# Configure index and key config
ev.init_index_config(
    key_path=key_path,
    key_id=key_id,
    eval_mode="mms32",
    preset="ip2",
    auto_key_setup=False,
    metadata_encryption=True,
)

# Register and load the key on the server
ev.register_key(key_id=key_id)
ev.load_key(key_id=key_id)

We can inspect the information about keys:
- `get_key_list()`: offers the list of registered key IDs
- `get_key_info()`: returns parameters for a specific key


In [ ]:
ev.get_key_list()

In [ ]:
ev.get_key_info(key_id)

### 3. Create Index

For ANN, we create an index with specified indexing parameters.
For IVF-VCT, `nlist`, `default_nprobe`, and `centroids` should be specified.


In [ ]:
# Configure index parameters
index_params = {
    "index_type": "IVF_VCT",
    "nlist": nlist,
    "default_nprobe": 1,
    "centroids": centroids.tolist(),
}

# Create index
index = ev.create_index(
    index_name,
    dim=dim,
    index_params=index_params,
)

### Encrypt Items

Before sending data to the server, we encrypt vectors and metadata on the client side using two different cryptosystems.

**Vectors (FHE):** the `Cipher` class provides a unified interface for both encryption and decryption of vectors and scores. Initialize it with the dimension and one or more key paths (`enc_key_path` for encryption, `sec_key_path` for decryption), or pass the key paths directly to each `encrypt` / `decrypt` call. `cipher.encrypt(v, "item")` encodes a vector for insertion and returns a `CipherBlock`.

For `IVF_VCT`, you should provide `centroids_idx` (cluster assignment per vector) during encryption, for example: `cipher.encrypt(vectors, centroids_idx=centroids_idx)`.

**Metadata (AES-256):** when metadata is sensitive, encrypt each entry on the client with `encrypt_metadata`, passing the metadata encryption key file (`MetadataKey.json`). The function serializes dict/list inputs to JSON and returns a Base64-encoded ciphertext string suitable for storage on the server.

Note that `Index.insert()` can also accept plaintext directly and perform the FHE encryption internally using the `Cipher` bound to the index — the explicit `encrypt` / `encrypt_metadata` calls below are shown to make the encryption step visible.

For more details, see [Encryption](https://docs.envector.io/user-guide/advanced-user-guide/cipher/encryption) in [enVector Documents](https://docs.envector.io/).


In [ ]:
from pyenvector.crypto import Cipher
from pyenvector.utils.aes import encrypt_metadata

enc_key_path = f"{key_path}/{key_id}/EncKey.json"
metadata_key_path = f"{key_path}/{key_id}/MetadataKey.json"

cipher = Cipher(dim=dim, enc_key_path=enc_key_path)

# Explicitly encrypt each item (vector) before sending
enc_vectors = cipher.encrypt(vectors, centroids_idx=centroids_idx)
print("Encrypted Vectors", enc_vectors)

# Explicitly encrypt metadata
enc_metadata = encrypt_metadata(metadata, key_path=metadata_key_path)
print("Encrypted Metadata", enc_metadata)

### Insert the Encrypted Vectors

Now we insert the data into the index on the enVector server. The SDK accepts either pre-encrypted `CipherBlock` inputs or plaintext (which it encrypts internally with the index's `Cipher`); we pass plaintext here for brevity.

`Index.insert()` is asynchronous on the server side, and a single insert flows through three server-side stages:
1. **flush** — the client splits the batch and the server persists each ciphertext shard.
2. **segmentation** — the server merges the persisted shards (identified by `request_ids`) into a search-ready segment.
3. **load** — the server loads the new segment into memory so that it becomes searchable.

By default a single `index.insert()` call runs all three stages and returns once they complete (or you can use `await_completion=True` to block on a specific stage).
- `index.insert(..., execute_until="flush", load=False, await_completion=False)` performs only the flush stage and captures the per-shard request IDs.
- `index.indexing(request_ids)` then triggers the segmentation merge for those request IDs.
- `index.load()` finally loads the merged segment into memory.

For more details, see [Insert](https://docs.envector.io/user-guide/insert) in [enVector Documents](https://docs.envector.io/).

In [ ]:
request_ids = []
index.insert(
    vectors,
    metadata,
    request_ids=request_ids,
    execute_until="flush",
    load=False,
    await_completion=False,
)
index.indexing(request_ids)
index.load()

### Encrypted search on the index

With the encrypted index loaded, we can perform similarity search directly over the ciphertexts.

The high-level `Index.search()` chains the four underlying APIs internally — `scoring`, `decrypt_score`, `get_topk_metadata_results`, and `decrypt_metadata` — but here we run each of them explicitly to see what happens at every stage.

`Index.scoring()` is the homomorphic part: the server computes inner products between the query and every encrypted vector in the index and returns one encrypted score block per query as a list of `CipherBlock`.

For more details, see [Scoring](https://docs.envector.io/user-guide/search/scoring) in the [enVector Documents](https://docs.envector.io/).

In [ ]:
query = vectors[0]
search_index = ev.Index(index_name)

# Server performs encrypted similarity scoring; response is encrypted
encrypted_scores = search_index.scoring(
    query,
    search_params={"nprobe": 16},
)[0]
encrypted_scores

### Decrypt search results

Because the server only saw ciphertexts, we need to decrypt the encrypted scores on the client to recover the similarity score values.
`Cipher.decrypt_score()` uses the secret key (`SecKey.json`) to produce a `"score"` array and `"shard_idx"` for multi-shard indexes.

For more details, see the [Decrypting Scores](https://docs.envector.io/user-guide/search/decrypting-scores) in the [enVector Documents](https://docs.envector.io/).

In [ ]:
sec_key_path = f"{key_path}/{key_id}/SecKey.json"
scores = cipher.decrypt_score(encrypted_scores, sec_key_path=sec_key_path)
scores


### Extract Top-k Relevance

With the decrypted scores in hand, the client picks the top-k indices and asks the server to return the metadata at those indices.
`Index.get_topk_metadata_results()` does both: it ranks the scores client-side and fetches the corresponding (still-encrypted with AES) metadata payloads from the server, returning a list of `{id, score, metadata}` dicts, which is decrypted with `MetadataKey.json` by client-side.

For more details, see the [Retrieving Top-k Metadata](https://docs.envector.io/user-guide/search/retrieving-top-k-metadata) in the [enVector Documents](https://docs.envector.io/).

In [ ]:
topk = search_index.get_topk_metadata_results(scores, top_k=3, output_fields=["metadata"])
print(*topk, sep="\n")

### Clean up

When the index and key are no longer needed, drop the index from the server with `ev.drop_index()` and remove the registered evaluation key with `ev.delete_key()`. The local key files under `key_path` are kept untouched — delete them manually if you no longer need to decrypt previously stored ciphertexts.

In [ ]:
ev.drop_index(index_name)
ev.delete_key(key_id)

For more detailed API reference, see [enVector Documents](https://docs.envector.io/).
